# AgentRegistry, end to end: scaffold a dice agent → run it on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer**. Run one cell at a time like a terminal. We scaffold a dice agent with `arctl`, run it locally, publish it, then deploy **the same agent** to two runtimes — Solo Enterprise for **kagent** (local kind) and **AWS Bedrock AgentCore** — by changing one line, the Deployment's `runtimeRef`.

> **Kernel:** pick **AgentCore demo (Python 3.13)** (top-right). Engineer setup lives in `setup/` (see `setup/README.md`) and is done before this.

## Connect to the platform

Loads `setup/.env.local`, puts `arctl` on the path, and mints a catalog token. Every later cell is a shell command (`!…`) that inherits this environment.

In [ ]:
import os, json, urllib.request, urllib.parse, pathlib
# work from the demo folder regardless of where the kernel started
if pathlib.Path('agentregistry-agentcore-kind/setup').is_dir(): os.chdir('agentregistry-agentcore-kind')
def _load(p):
    if p and os.path.exists(p):
        for ln in open(p):
            ln = ln.strip()
            if ln.startswith('export '): ln = ln[7:]
            if ln and not ln.startswith('#') and '=' in ln:
                k, v = ln.split('=', 1); os.environ[k] = v.strip().strip('"').strip("'")
_load('setup/.env.local'); _load(os.environ.get('SECRETS_FILE', ''))
os.environ['PATH'] = os.path.expanduser('~/.arctl/bin') + os.pathsep + os.environ.get('PATH', '')
os.environ.setdefault('ARCTL_API_BASE_URL', 'http://localhost:12121')
os.environ.setdefault('CLUSTER_NAME', 'agentcore-demo')
os.environ['NO_COLOR']='1'; os.environ['TERM']='dumb'  # clean output (no terminal-probe escapes)
try:
    _d = urllib.parse.urlencode({'grant_type':'client_credentials','client_id':'admin','scope':'openid profile email Groups'}).encode()
    os.environ['ARCTL_API_TOKEN'] = json.load(urllib.request.urlopen(os.environ['ARCTL_API_BASE_URL'] + '/api/autoauth/oauth/token', _d, timeout=10))['access_token']
except Exception as e:
    print('token error:', e)
print('token:', 'ok' if os.environ.get('ARCTL_API_TOKEN') else 'MISSING',
      '| anthropic:', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING',
      '| cluster: kind-' + os.environ['CLUSTER_NAME'])

In [ ]:
!arctl get runtimes

## 1. Create a new agent project

Scaffolds the dice-rolling agent into `agentdemo/`. **Open it in the Explorer** to walk through `roll_die` / `check_prime`.

In [ ]:
!arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5

## 2. Walk through the dice agent

In [ ]:
!cat agentdemo/agentdemo/agent.py

## 3. Build the agent image

In [ ]:
!arctl build ./agentdemo

Run it locally in an interactive chat (use a **terminal** — it's interactive):

```sh
arctl run ./agentdemo
```

## 4. Publish to the catalog

In [ ]:
!arctl build ./agentdemo --push

In [ ]:
!arctl apply -f agentdemo/agent.yaml

In [ ]:
!arctl get agent agentdemo

## 5. Deploy the agent onto kagent (runtime #1)

`runtimeRef: kind-kagent` is **the one line that changes for AWS later.** (The runtime was registered during setup.)

In [ ]:
!envsubst < setup/yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
!arctl get deployments

## 6. Talk to the dice agent — through real OIDC

Mints a real Keycloak token for **alice** and sends an A2A message. Watch it call `roll_die` then `check_prime`.

In [ ]:
!./setup/scripts/ask.sh "Roll a 20-sided die and tell me whether the result is a prime number."

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

Identical agent, deployed to AWS — native Bedrock Claude via the AWS role (no API key). **Needs an AWS account; skip for a local-only demo.**

## 7. Sign in to AWS

Runs `aws sso login` and hands the credentials to the arctl daemon, then loads them into this session so the next cells can deploy.

In [ ]:
import subprocess, os
_r = subprocess.run('source setup/scripts/aws-login.sh', shell=True, executable='/bin/bash',
                    capture_output=True, text=True)
print(_r.stdout.strip() or _r.stderr.strip()[-400:])
# capture the AWS creds + token the script set, into this session
_e = subprocess.run('source setup/scripts/aws-login.sh >/dev/null 2>&1; env', shell=True, executable='/bin/bash',
                    capture_output=True, text=True)
for ln in _e.stdout.splitlines():
    if '=' in ln:
        k, v = ln.split('=', 1)
        if k in ('AWS_REGION','AWS_ACCOUNT_ID','AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','ARCTL_API_TOKEN','DOCKER_REPO'):
            os.environ[k] = v

## 8. Deploy the same agent to AgentCore

One command: makes the agent multi-cloud, grants the cross-account role, registers the `BedrockAgentCore` runtime, pushes image + source, deploys, waits for READY.

In [ ]:
!./setup/scripts/agentcore-deploy.sh

## 9. Test the dice agent on AgentCore

In [ ]:
!./setup/scripts/ac-invoke.sh "Roll a 20-sided die and tell me whether the result is a prime number."

**The takeaway:** one agent, scaffolded with `arctl`, published once, ran unchanged on Kubernetes (kagent) *and* AWS Bedrock AgentCore.

## Reset / teardown

```sh
./setup/scripts/reset.sh      # back to start; platform stays up
./setup/scripts/cleanup.sh    # full teardown
```